# Annotation consensus

Load `raw_data/image/to_annotate.tif` and the annotator masks in `raw_data/labels`. Predict nuclei with StarDist and Cellpose 3 on the CPU, merge annotators and predictions into one `{stack, names}` bundle, align ids to the first annotator mask, and review everything in Napari.

Label files are read in filename order. The first annotator file is the reference.

In [ ]:
from pathlib import Path
import numpy as np

import utils

ROOT = Path.cwd()
IMAGE_PATH = ROOT / "raw_data" / "image" / "to_annotate.tif"
LABELS_DIR = ROOT / "raw_data" / "labels"
PREDICTIONS_DIR = ROOT / "predictions"

utils.use_cpu()
image = utils.load_image(IMAGE_PATH)
image.shape, image.dtype

## Predict nuclei

StarDist uses the fluorescent-nuclei model. Cellpose 3 uses its `nuclei` model. Both run without a GPU. Predictions are written to `predictions/` and reused from disk on later runs (`force=True` recomputes them).

In [ ]:
predictions = utils.predict_nuclei(image, IMAGE_PATH.name, PREDICTIONS_DIR)
{name: labels.max() for name, labels in predictions.items()}

## Collect the annotator labels

Annotator masks are loaded as a `LabelBundle`: a `stack` array plus the source `names`. Planes that are not 2D, or whose shape differs from the first file, are omitted.

In [ ]:
annotator_labels = utils.load_label_stack(LABELS_DIR)
annotator_labels["stack"].shape, annotator_labels["names"]

## Merge predictions into the label stack

Predictions are converted to the same `{stack, names}` bundle, then concatenated with the annotator labels. Filenames stay paired with their planes for alignment and Napari.

In [ ]:
prediction_labels = utils.label_bundle_from_mapping(predictions)
labels = utils.combine_label_bundles(annotator_labels, prediction_labels)
labels["stack"].shape, labels["names"]

## Align label ids

The first plane in `labels` (first annotator file) is relabeled to `1..N` and used as the reference. Every other plane—annotators and predictions—is kept only when it has the same number of objects. Matching assigns each object the id of one reference object by minimizing centroid distance, so the same id is the same nucleus across sources.

In [ ]:
aligned = utils.align_label_stack(labels)
aligned["stack"].shape, aligned["names"]

## Inter-annotation uncertainty

Align the annotator masks alone (predictions stay out of this average). For each object id, binarize every plane to `{0, 1}`, average across annotators, then take the per-pixel maximum over objects. Values stay in `[0, 1]`: high = agreement, mid = disagreement, low = background. Napari shows this map with the inferno colormap.

In [ ]:
annotator_aligned = utils.align_label_stack(annotator_labels)
agreement = utils.label_agreement_map(annotator_aligned)
agreement.min(), agreement.max(), agreement.shape

masked = np.where(agreement == 1, 0.0, agreement)  # full agreement → 0
uncertainty = 1.0 - masked
uncertainty = np.where(uncertainty == 1, 0.0, uncertainty)  # former background → 0

## Review in Napari

The image layer uses the original filename. The uncertainty layer uses inferno (`[0, 1]` agreement). The aligned annotator and prediction masks are one stack: the slider selects a source and the layer name switches to that filename.

In [ ]:
viewer = utils.show_in_napari(
    image,
    IMAGE_PATH.name,
    aligned
)
[layer.name for layer in viewer.layers]
viewer.add_image(uncertainty, colormap="inferno")

In [ ]:
# Height field: each (y, x) fills z-planes [0, height)
heights = np.clip(np.rint(np.asarray(uncertainty, dtype=float) * 100), 0, 100).astype(np.int32)
z = np.arange(100, dtype=np.int32)[:, None, None]
# Voxel value = depth (1..100) so the colormap reads as a depth map; 0 = empty
uncertainty_volume = np.where(z < heights, z + 1, 0).astype(np.float32)
uncertainty_volume.shape

layer = viewer.add_image(
    uncertainty_volume,
    name="uncertainty_volume",
    colormap="inferno",
    rendering="mip",
    opacity=0.5,
    contrast_limits=(0, 100),
)
layer.colorbar.visible = True

# 3D camera
viewer.dims.ndisplay = 3
cam = viewer.camera
cam.orientation = ("towards", "down", "right")
cam.zoom = 2.72
cam.perspective = 45
cam.angles = (-22, -7, -35)  # Z, Y, X
cam.synced = True

uncertainty_volume.shape, uncertainty_volume.min(), uncertainty_volume.max()

In [ ]:
fig = utils.uncertainty_bars_3d(uncertainty*100, threshold=0.0, stride=1)
fig.show()